<a href="https://colab.research.google.com/github/YukinoshitaSherry/CSCI572-Information_Retrieval_And_Web_Search_Engines/blob/master/singleCell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import rbf_kernel

In [ ]:
# View Formats
file_path = r'/content/AissaBenevolenskaya2021.h5ad'
adata = sc.read_h5ad(file_path)

In [ ]:
# Preprocessing
def load_and_preprocess_data(file_path):
    adata = sc.read_h5ad(file_path)
    print(adata)

    # 过滤低质量细胞
    sc.pp.filter_cells(adata, min_counts=500)
    sc.pp.filter_cells(adata, min_genes=750)

    # 过滤低质量基因
    sc.pp.filter_genes(adata, min_cells=100)

    # 数据标准化
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    X = adata.X.toarray()
    y = adata.obs[['ncounts', 'ngenes', 'percent_mito', 'percent_ribo']].values
    return X, y

In [ ]:
# split Data Set
X, y = load_and_preprocess_data(file_path)
# 分割数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

AnnData object with n_obs × n_vars = 119071 × 17820
    obs: 'GEO', 'time', 'cell_line', 'perturbation', 'batch', 'subseries', 'replicate', 'tissue_type', 'cancer', 'perturbation_type', 'disease', 'celltype', 'organism', 'nperts', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'chembl-ID'
    var: 'mt', 'ribo', 'ncounts', 'ncells'


In [ ]:
#Evaluate Matrix
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    # 对每个目标变量计算 Pearson 相关系数
    pearson_corr = np.array([pearsonr(y_true[:, i], y_pred[:, i])[0] for i in range(y_true.shape[1])])
    mmd_value = md_rbf_kernel(y_true, y_pred)
    return mse, pearson_corr, mmd_value

def mmd_rbf(X, Y, gamma=1.0):
    K_XX = rbf_kernel(X, X, gamma=gamma)
    K_YY = rbf_kernel(Y, Y, gamma=gamma)
    K_XY = rbf_kernel(X, Y, gamma=gamma)
    mmd = np.mean(K_XX) + np.mean(K_YY) - 2 * np.mean(K_XY)
    return mmd

In [ ]:
#XGBoost
import xgboost as xgb

# 将数据转换为 DMatrix 格式
# 训练数据
dtrain = xgb.DMatrix(X_train, label=y_train)
# 测试数据
dtest = xgb.DMatrix(X_test, label=y_test)

# 设置 XGBoost 模型的超参数
params = {
    'objective': 'reg:squarederror',
    'max_depth': 3,
    'eta': 0.05,
    'eval_metric': 'rmse',
    'lambda': 2,
    'alpha': 0.5
}

# 设定训练轮数
num_round = 200

# 定义评估数据集
evals = [(dtrain, 'train'), (dtest, 'eval')]

# 训练 XGBoost 模型，加入 early_stopping_rounds早停机制，防止过拟合
bst = xgb.train(params, dtrain, num_boost_round=num_round, evals=evals, early_stopping_rounds=50)

# 使用训练好的模型对测试集进行预测
y_pred = bst.predict(dtest)

# 评估模型性能
mse, pearson_corr, mmd_value = evaluate_model(y_test, y_pred)
print(f"XGBoost - MSE: {mse}, Pearson Correlation: {pearson_corr}, MMD value: {mmd_value}")

[0]	train-rmse:1802.38235	eval-rmse:1687.37822
[1]	train-rmse:1740.68905	eval-rmse:1624.24397
[2]	train-rmse:1685.70425	eval-rmse:1569.43516
[3]	train-rmse:1628.05660	eval-rmse:1517.61303
[4]	train-rmse:1572.12862	eval-rmse:1467.34941
[5]	train-rmse:1521.91381	eval-rmse:1420.05506
[6]	train-rmse:1474.01550	eval-rmse:1377.43599
[7]	train-rmse:1425.98032	eval-rmse:1334.02344
[8]	train-rmse:1381.36750	eval-rmse:1291.62015
[9]	train-rmse:1339.62859	eval-rmse:1254.55864
[10]	train-rmse:1298.16796	eval-rmse:1216.64259
[11]	train-rmse:1258.73040	eval-rmse:1178.77011
[12]	train-rmse:1221.28020	eval-rmse:1145.96807
[13]	train-rmse:1183.82755	eval-rmse:1113.15310
[14]	train-rmse:1149.33603	eval-rmse:1083.55764
[15]	train-rmse:1116.29344	eval-rmse:1052.52686
[16]	train-rmse:1083.68508	eval-rmse:1017.40126
[17]	train-rmse:1054.05418	eval-rmse:989.21421
[18]	train-rmse:1024.26394	eval-rmse:962.08503
[19]	train-rmse:995.07451	eval-rmse:937.73775
[20]	train-rmse:968.11865	eval-rmse:911.45986
[21]	tra

NameError: name 'md_rbf_kernel' is not defined

In [ ]:
#Random Forest
# 定义随机森林回归模型
rf_model = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_split=2, random_state=42)
# 训练模型
rf_model.fit(X_train, y_train)
# 预测测试集
y_pred = rf_model.predict(X_test)

# 评估模型性能
mse, pearson_corr, mmd_value = evaluate_model(y_test, y_pred)
print(f"Random Forest - MSE: {mse}, Pearson Correlation: {pearson_corr}, MMD value: {mmd_value}")

Random Forest - MSE: 155451.83904925504, Pearson Correlation: [0.96391647 0.97563953 0.4623337  0.58798882], MMD value: 0.0017929372728742296


In [ ]:
#Linear Regression
# 定义线性回归模型
lr_model = LinearRegression()

# 训练模型
lr_model.fit(X_train, y_train)

# 预测测试集
y_pred = lr_model.predict(X_test)

# 评估模型性能
mse, pearson_corr, mmd_value = evaluate_model(y_test, y_pred)
print(f"Linear Regression - MSE: {mse}, Pearson Correlation: {pearson_corr}, MMD value: {mmd_value}")

In [ ]:
#VAE
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# 转换为 PyTorch tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# 创建 DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# VAE模型
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, output_dim):
        super(VAE, self).__init__()

        # 编码器部分
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc21 = nn.Linear(256, latent_dim)  # 均值
        self.fc22 = nn.Linear(256, latent_dim)  # 对数方差

        # 解码器部分
        self.fc3 = nn.Linear(latent_dim, 256)
        self.fc4 = nn.Linear(256, output_dim)

    def encode(self, x):
        h1 = torch.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = torch.relu(self.fc3(z))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def vae_loss(recon_x, x, mu, logvar):
    # 重构损失
    BCE = nn.MSELoss()(recon_x, x)
    kl_divergence = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return BCE + kl_divergence

# 定义模型，优化器
input_dim = X_train.shape[1]  # 输入维度，通常是基因表达矩阵的列数
latent_dim = 10  # 潜在空间维度，可以调整
output_dim = y_train.shape[1]  # 输出维度，通常是目标变量的数量

vae = VAE(input_dim=input_dim, latent_dim=latent_dim, output_dim=output_dim)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

# 训练过程
epochs = 50
for epoch in range(epochs):
    vae.train()
    train_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        recon_batch, mu, logvar = vae(data)
        loss = vae_loss(recon_batch, target, mu, logvar)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # 训练集的损失打印
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss / len(train_loader)}')

    # 每个epoch结束时，评估模型在测试集上的表现
    vae.eval()
    y_pred = []
    y_true = []

    with torch.no_grad():
        for data, target in test_loader:
            recon_batch, mu, logvar = vae(data)
            y_pred.append(recon_batch.numpy())  # 预测的重构数据
            y_true.append(target.numpy())  # 真实数据

    y_pred = np.vstack(y_pred)
    y_true = np.vstack(y_true)

    # 计算 MSE, Pearson, MMD
    mse, pearson_corr, mmd_value = evaluate_model(y_true, y_pred)

    # 输出评估结果
    print(f'Epoch {epoch+1} Evaluation:')
    print(f'MSE: {mse}')
    print(f'Pearson Correlation: {pearson_corr}')
    print(f'MMD: {mmd_value}')



In [ ]:
#scGen
!pip install scgen

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.3/459.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.8/360.8 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.2 MB/s eta 0:00:00

In [ ]:
!pip show scvi-tools

Name: scvi-tools
Version: 1.2.2.post2
Summary: Deep probabilistic analysis of single-cell omics data.
Home-page: https://scvi-tools.org
Author: The scvi-tools development team
Author-email: 
License: BSD 3-Clause License

Copyright (c) 2024, Adam Gayoso, Romain Lopez, Martin Kim, Pierre Boyeau, Nir Yosef

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS S

In [ ]:
import logging
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.metrics.pairwise import rbf_kernel
from scgen import SCGEN  # 导入 scGen


# 将数据转换为 AnnData 对象（scGen 需要 AnnData 格式）
adata = sc.AnnData(X_train)
adata.obs['condition'] = ['control'] * len(y_train)  # 假设所有数据都是控制组
adata.obs['ncounts'] = y_train[:, 0]
adata.obs['ngenes'] = y_train[:, 1]
adata.obs['percent_mito'] = y_train[:, 2]
adata.obs['percent_ribo'] = y_train[:, 3]

ModuleNotFoundError: No module named 'scvi._compat'

In [ ]:
# 初始化 scGen 模型
scgen_model = SCGEN(adata)

# 训练模型
model.train(
    max_epochs=100,
    batch_size=32,
    early_stopping=True,
    early_stopping_patience=25
)

scgen_model.save("/content/saved_models/model_perturbation_prediction.pt", overwrite=True)
# 生成预测结果
# 假设你有一个控制组（control）和一个扰动组（perturbed）
control_key = "control"
perturbed_key = "perturbed"

# 生成控制组的预测扰动结果
predicted_adata = scgen_model.predict(adata, control_key=control_key, perturbed_key=perturbed_key)

# 提取真实数据和预测数据
true_data = adata[adata.obs["condition"] == perturbed_key].X
predicted_data = predicted_adata.X

# 计算 MSE、皮尔森相关度和 MMD
mse, pearson_corr, mmd_value = evaluate_model(true_data, predicted_data)

print(f"scGen - MSE: {mse}")
print(f"scGen - Pearson Correlation: {pearson_corr}")
print(f"scGen - MMD value: {mmd_value}")


NameError: name 'SCGEN' is not defined

In [ ]:
from gears import PertData, GEARS
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr

In [ ]:
# Names of the scGenePT models to load. Note that these have to match the keys in the model_name2model_variation dict
models = ['scgpt', 'scgenept_go_c_gpt_concat', 'scgenept_ncbi+uniprot_gpt']
# models = ['scgpt',  'scgenept_go_c_gpt_concat', 'scgenept_go_p_gpt_concat', 'scgenept_go_f_gpt_concat', 'scgenept_go_all_gpt_concat', 'scgenept_ncbi_gpt', 'scgenept_ncbi+uniprot_gpt']
trained_models = {}

In [ ]:
import scanpy as sc
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Load dataset
adata = sc.read_h5ad('GSM_raw.h5ad')

# Normalization and HVG selection
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]

# PCA on pre-perturbation data
pca = PCA(n_components=50)
X_pre = pca.fit_transform(adata.X.toarray())

# Assuming post-perturbation data is in a separate AnnData object
X_post = pca.transform(adata_post.X.toarray())

# Train-validation-test split
X_train, X_test, y_train, y_test = train_test_split(X_pre, X_post, test_size=0.2)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25)


ImportError: cannot import name '_add_to_diagonal' from 'sklearn.utils._array_api' (/usr/local/lib/python3.11/dist-packages/sklearn/utils/_array_api.py)

In [ ]:
import optuna
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error



def objective(trial):
    # 超参数搜索空间
    params = {
        'tree_method': 'gpu_hist',  # 使用GPU加速
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.5, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.3),
        'n_estimators': 2000,  # 使用正确的参数名
        'early_stopping_rounds': 50  # 提前停止参数
    }

    # 创建多输出回归模型
    model = MultiOutputRegressor(
        xgb.XGBRegressor(**params),
        n_jobs=-1  # 启用并行
    )

    # 训练模型（注意移除非法的num_boost_round参数）
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # 预测验证集
    pred = model.predict(X_val)
    return mean_squared_error(y_val, pred)

# 创建Optuna研究
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# 运行优化
study.optimize(objective, n_trials=100, show_progress_bar=True)

# 输出最佳参数
best_params = study.best_params
print(f"Best parameters: {best_params}")


[I 2025-02-19 12:21:47,151] A new study created in memory with name: no-name-8dd49d46-1c44-444f-a7c9-0be8b622bb1c


  0%|          | 0/100 [00:00<?, ?it/s]

[W 2025-02-19 12:22:00,983] Trial 0 failed with parameters: {'max_depth': 5, 'learning_rate': 0.36808608148776095, 'subsample': 0.892797576724562, 'reg_alpha': 0.6387926357773329, 'reg_lambda': 0.24041677639819287, 'gamma': 0.04679835610086079} because of the following error: XGBoostError('[12:22:00] /workspace/src/objective/regression_obj.cu:50: Check failed: info.labels.Size() == preds.Size() (1420 vs. 5680) : Invalid shape of labels.\nStack trace:\n  [bt] (0) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x25c1ac) [0x7a621ca5c1ac]\n  [bt] (1) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xd31b8c) [0x7a621d531b8c]\n  [bt] (2) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xd5b44f) [0x7a621d55b44f]\n  [bt] (3) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x5f8c53) [0x7a621cdf8c53]\n  [bt] (4) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(XGBoosterUpdateOneIter+0x6f) [0x7a621c965a1

XGBoostError: [12:22:00] /workspace/src/objective/regression_obj.cu:50: Check failed: info.labels.Size() == preds.Size() (1420 vs. 5680) : Invalid shape of labels.
Stack trace:
  [bt] (0) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x25c1ac) [0x7a621ca5c1ac]
  [bt] (1) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xd31b8c) [0x7a621d531b8c]
  [bt] (2) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xd5b44f) [0x7a621d55b44f]
  [bt] (3) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x5f8c53) [0x7a621cdf8c53]
  [bt] (4) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(XGBoosterUpdateOneIter+0x6f) [0x7a621c965a1f]
  [bt] (5) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x7a6243caee2e]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x7a6243cab493]
  [bt] (7) /usr/lib/python3.11/lib-dynload/_ctypes.cpython-311-x86_64-linux-gnu.so(+0xa4d8) [0x7a6243cbe4d8]
  [bt] (8) /usr/lib/python3.1

In [ ]:
# Train final model
final_model = MultiOutputRegressor(
    xgb.XGBRegressor(**best_params, tree_method='gpu_hist', n_estimators=1000)
)
final_model.fit(X_train, y_train)

# Predict and reconstruct gene expressions
y_pred = final_model.predict(X_test)
gene_pred = pca.inverse_transform(y_pred)
gene_true = pca.inverse_transform(y_test)

# Metrics
mse = mean_squared_error(gene_true, gene_pred)
pearson_corr = np.corrcoef(gene_true.flatten(), gene_pred.flatten())[0, 1]

# MMD Calculation
from sklearn.gaussian_process.kernels import RBF
def compute_mmd(x, y, kernel=RBF(1.0)):
    Kxx = kernel(x, x)
    Kyy = kernel(y, y)
    Kxy = kernel(x, y)
    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()

mmd_score = compute_mmd(gene_true, gene_pred)

In [ ]:
print(f"XGBoost MSE：{mse:.4f},pearson:{pearson_corr:.4f},mmd:{mmd_score:.4f}")